# 01 - EchoNet-Dynamic dataset exploration

This notebook validates the raw EchoNet-Dynamic layout before any segmentation preprocessing. `FileList.csv` stores one row per video and includes metadata such as EF, frame size, frame count, FPS, and the official split. `VolumeTracings.csv` stores expert LV tracing coordinates for selected frames; these are coordinates, not masks.

In [ ]:
# Kaggle execution order: run this notebook first to verify paths and tracing structure.
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import pandas as pd

from src.utils import load_echonet_tables, tracing_group_to_polygon, video_path_from_name

RAW_DIR = PROJECT_ROOT / "data" / "raw" / "EchoNet-Dynamic"
VIDEOS_DIR = RAW_DIR / "Videos"
FIGURES_DIR = PROJECT_ROOT / "outputs" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

file_list, tracings = load_echonet_tables(RAW_DIR)

## Dataset statistics

The basic checks below confirm the number of videos, split distribution, frame geometry, and tracing coverage. EchoNet-Dynamic videos are usually 112 x 112 pixels, but the code keeps the dimensions data-driven.

In [ ]:
display(file_list.head())
display(tracings.head())

print(f"FileList rows: {len(file_list):,}")
print(f"Tracing rows: {len(tracings):,}")
print(f"Unique traced videos: {tracings['FileName'].nunique():,}")
print(f"Unique traced video-frame pairs: {tracings.groupby(['FileName', 'Frame']).ngroups:,}")

display(file_list['Split'].value_counts(dropna=False).rename('videos_by_split'))
display(file_list[['FrameHeight', 'FrameWidth', 'FPS', 'NumberOfFrames']].describe())
display(tracings[['X1', 'Y1', 'X2', 'Y2', 'Frame']].describe())

## Verify video locations

`VolumeTracings.csv` includes `.avi` in `FileName`, while `FileList.csv` typically stores video stems without the suffix. The helper normalizes both forms.

In [ ]:
sample_video_names = file_list['FileName'].head(10).tolist()
located = [video_path_from_name(name, RAW_DIR).exists() for name in sample_video_names]
display(pd.DataFrame({'FileName': sample_video_names, 'video_found': located}))

all_video_paths = [video_path_from_name(name, RAW_DIR) for name in file_list['FileName']]
missing = [p.name for p in all_video_paths if not p.exists()]
print(f"Videos referenced by FileList but missing on disk: {len(missing):,}")
print(missing[:10])

## Inspect tracing coordinates

For each traced frame, EchoNet stores multiple rows of paired points. `(X1, Y1)` follows one side of the LV border and `(X2, Y2)` follows the opposite side. A segmentation polygon is created by walking down the first side and returning along the reversed second side.

In [ ]:
group_key, group_rows = next(iter(tracings.groupby(['FileName', 'Frame'])))
file_name, frame_idx = group_key
polygon = tracing_group_to_polygon(group_rows)
closed_polygon = pd.DataFrame(polygon, columns=['x', 'y'])

print(f"Sample traced frame: {file_name}, frame {frame_idx}")
display(group_rows.head())
display(closed_polygon.head())

plt.figure(figsize=(4, 4))
plt.scatter(group_rows['X1'], group_rows['Y1'], label='X1/Y1 side', s=16)
plt.scatter(group_rows['X2'], group_rows['Y2'], label='X2/Y2 side', s=16)
plt.plot(polygon[:, 0], polygon[:, 1], color='black', linewidth=1)
plt.gca().invert_yaxis()
plt.axis('equal')
plt.title('Sample LV tracing coordinates')
plt.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'sample_tracing_coordinates.png', dpi=150, bbox_inches='tight')
plt.show()

## Key observations

- The raw dataset contains videos plus two CSV tables: video-level metadata and frame-level tracing coordinates.
- Tracing coordinates must be rasterized into binary masks before segmentation training.
- A single video can have multiple traced frames, commonly ED/ES frames.
- The official EchoNet split can be reused after preprocessing because output sample names keep the original video stem.